In [ ]:
#Download Data
Data URL - https://krunal-launchpad.s3.ap-south-1.amazonaws.com/data/factOrders.csv

In [1]:
#Create a SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark=SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/04 03:49:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#Read data & create dataframe
df=spark.read.csv("factOrders.csv",header=True,inferSchema=True)
df.show()

+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+
|OrderID| DateKey|CustomerKey|RestaurantKey|DeliveryKey|ItemKey|Quantity|TotalAmount|DiscountPerentage|OrderStatus|
+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+
|      1|20240901|        145|          241|        345|    443|       2|        952|              5.0|  Completed|
|      2|20240901|        148|          214|        303|    402|       1|        827|              0.0|    Pending|
|      3|20240902|        101|          231|        348|    421|       3|       1921|             10.0|  Completed|
|      4|20240902|        104|          225|        328|    437|       2|       1484|              5.0|  Cancelled|
|      5|20240903|        104|          203|        322|    405|       1|       1103|              0.0|  Completed|
|      6|20240903|        140|          204|        340|    417|       4

In [3]:
#Find the total number of orders
df.count()

150

In [4]:
#Find min, max and average order value. Return as dictionary

metrics_df = df.select(
    F.min("TotalAmount").alias("min_order_value"),
    F.max("TotalAmount").alias("max_order_value"),
    F.avg("TotalAmount").alias("avg_order_value")
)

result_dict = metrics_df.first().asDict()

result_dict

{'min_order_value': 291,
 'max_order_value': 2230,
 'avg_order_value': 1235.1333333333334}

In [5]:
#Find the number different order status
df.select("OrderStatus").distinct().show()

df.groupby("OrderStatus").count().show()

+-----------+
|OrderStatus|
+-----------+
|  Completed|
|  Cancelled|
| In Transit|
|  Delivered|
|    Pending|
+-----------+

+-----------+-----+
|OrderStatus|count|
+-----------+-----+
|  Completed|   96|
|  Cancelled|   21|
| In Transit|    6|
|  Delivered|    6|
|    Pending|   21|
+-----------+-----+



In [6]:
#Find the % of orders completed
completed=df.filter(F.col("OrderStatus")=="Completed").count()
total=df.count()
completed/total*100

64.0

In [7]:
#Find the % of orders pending
pending=df.filter(F.col("OrderStatus")=="Pending").count()
pending/total*100

14.000000000000002

In [8]:
#Calculate Total Sales
df1=df.filter((F.col("OrderStatus")=="Completed") | (F.col("OrderStatus")=="Delivered"))
df1.agg(F.sum("TotalAmount").alias("Total_sales")).show()

+-----------+
|Total_sales|
+-----------+
|     125003|
+-----------+



In [9]:
#Find the most popular restaurant [output should be restaurant ID as integer]
df.groupby("RestaurantKey").count().orderBy(F.col("count").desc()).first()[0]

204

In [12]:
#Add a new column "customeDiscount" with DiscountPercentage as 10 for order value above 1500 & 7 for all others
df=df.withColumn("customDiscount",F.when(F.col("TotalAmount")>1500,10).otherwise(7))
df.show()

+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+
|OrderID| DateKey|CustomerKey|RestaurantKey|DeliveryKey|ItemKey|Quantity|TotalAmount|DiscountPerentage|OrderStatus|customDiscount|
+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+
|      1|20240901|        145|          241|        345|    443|       2|        952|              5.0|  Completed|             7|
|      2|20240901|        148|          214|        303|    402|       1|        827|              0.0|    Pending|             7|
|      3|20240902|        101|          231|        348|    421|       3|       1921|             10.0|  Completed|            10|
|      4|20240902|        104|          225|        328|    437|       2|       1484|              5.0|  Cancelled|             7|
|      5|20240903|        104|          203|        322|    405|       1|       110

In [16]:
#Add a new column "effectivePrice" with customDiscount applied on TotalAmount
df=df.withColumn("effectivePrice",F.col("TotalAmount")*(F.lit(100)-F.col("customDiscount"))/100)
df.show()

+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+--------------+
|OrderID| DateKey|CustomerKey|RestaurantKey|DeliveryKey|ItemKey|Quantity|TotalAmount|DiscountPerentage|OrderStatus|customDiscount|effectivePrice|
+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+--------------+
|      1|20240901|        145|          241|        345|    443|       2|        952|              5.0|  Completed|             7|        885.36|
|      2|20240901|        148|          214|        303|    402|       1|        827|              0.0|    Pending|             7|        769.11|
|      3|20240902|        101|          231|        348|    421|       3|       1921|             10.0|  Completed|            10|        1728.9|
|      4|20240902|        104|          225|        328|    437|       2|       1484|              5.0|  Cancelled|         

In [19]:
#Add a new column to identify difference between TotalAmount & effectivePrice
df=df.withColumn("diff",F.col("TotalAmount")-F.col("effectivePrice"))
df.show()

+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+--------------+------------------+
|OrderID| DateKey|CustomerKey|RestaurantKey|DeliveryKey|ItemKey|Quantity|TotalAmount|DiscountPerentage|OrderStatus|customDiscount|effectivePrice|              diff|
+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+--------------+------------------+
|      1|20240901|        145|          241|        345|    443|       2|        952|              5.0|  Completed|             7|        885.36| 66.63999999999999|
|      2|20240901|        148|          214|        303|    402|       1|        827|              0.0|    Pending|             7|        769.11|57.889999999999986|
|      3|20240902|        101|          231|        348|    421|       3|       1921|             10.0|  Completed|            10|        1728.9| 192.0999999999999|
|      4|2

In [22]:
#Find the total profit loss if custom discount is applied
df.select("diff").show()

+------------------+
|              diff|
+------------------+
| 66.63999999999999|
|57.889999999999986|
| 192.0999999999999|
|103.88000000000011|
| 77.21000000000004|
| 72.16999999999996|
| 199.9000000000001|
| 165.0999999999999|
| 91.06999999999994|
|             201.5|
| 38.14999999999998|
| 204.5999999999999|
| 209.5999999999999|
|60.690000000000055|
| 95.33999999999992|
| 176.4000000000001|
| 60.75999999999999|
|168.79999999999995|
| 40.74000000000001|
| 68.11000000000001|
+------------------+
only showing top 20 rows


In [ ]:
#Find the most popular food item


In [26]:

df= df.withColumn(
    "actual_date", 
    F.to_date(F.col("DateKey"), "yyyyMMdd")
)

df.show()
df.printSchema()

+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+--------------+------------------+-----------+
|OrderID| DateKey|CustomerKey|RestaurantKey|DeliveryKey|ItemKey|Quantity|TotalAmount|DiscountPerentage|OrderStatus|customDiscount|effectivePrice|              diff|actual_date|
+-------+--------+-----------+-------------+-----------+-------+--------+-----------+-----------------+-----------+--------------+--------------+------------------+-----------+
|      1|20240901|        145|          241|        345|    443|       2|        952|              5.0|  Completed|             7|        885.36| 66.63999999999999| 2024-09-01|
|      2|20240901|        148|          214|        303|    402|       1|        827|              0.0|    Pending|             7|        769.11|57.889999999999986| 2024-09-01|
|      3|20240902|        101|          231|        348|    421|       3|       1921|             10.0|  Completed|

In [34]:
#Find the busiest Day
df.groupby("actual_date").count().orderBy(F.col("count").desc()).limit(1).show()

+-----------+-----+
|actual_date|count|
+-----------+-----+
| 2024-09-17|    9|
+-----------+-----+



In [41]:
#Find the busiest Month
df.select(F.month("actual_date").alias("Month")).groupby("Month").count().orderBy(F.col("count").desc()).limit(1).show()

+-----+-----+
|Month|count|
+-----+-----+
|    9|  150|
+-----+-----+



In [45]:
#Find the most profitable Day
df.groupby("actual_date").agg(F.sum("TotalAmount").alias("finalamount")).orderBy(F.col("finalamount").desc()).limit(1).show()

+-----------+-----------+
|actual_date|finalamount|
+-----------+-----------+
| 2024-09-16|      11634|
+-----------+-----------+



In [57]:
#Find the most profitable month
df1=df.withColumn("Month",F.month("actual_date"))

df1.groupby("Month").agg(F.sum("TotalAmount").alias("finalamount")).orderBy(F.col("finalamount").desc()).limit(1).show()

+-----+-----------+
|Month|finalamount|
+-----+-----------+
|    9|     185270|
+-----+-----------+



In [65]:
#Find the costliest item
df.withColumn("per_item_cost", F.col("TotalAmount")/F.col("Quantity")).orderBy("per_item_cost").limit(1).select("per_item_cost").show()

+-------------+
|per_item_cost|
+-------------+
|         71.8|
+-------------+



In [ ]:
#Find teh revenue lost (orders in transit)


In [ ]:
#Find most loyal customer
